# 04 — Sequential Graph + Tool-Level GRU

## Constat du notebook 03

Le graph de **co-occurrence** (Jaccard sur les sets de tools) est inutile :
- 14.5M edges pour 7117 workflows = quasi-clique
- Les méthodes graph ont **0% SeqAcc** partout
- Top-1 cosine reste le meilleur à toutes les échelles

## Hypothèse

Le graph de **transitions séquentielles** (tool_A → tool_B, extrait des DAGs n8n)
devrait être radicalement différent :
- **Dirigé** : A→B ≠ B→A
- **Sparse** : 1156 edges vs 14.5M (densité 2% vs 57%)
- **Sémantique** : capture l'ordre réel des opérations

## Plan

1. Construire les séquences ordonnées de MCP tools depuis les DAGs n8n (tier 1)
2. Construire le graph dirigé WF→WF (via transitions tool-level)
3. Méthodes graph dirigées vs flat
4. GRU entraîné sur les vraies séquences de tools
5. Comparaison à différentes échelles de catalog

In [ ]:
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from collections import defaultdict, Counter
from sklearn.metrics.pairwise import cosine_similarity
import time

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cpu")

DATA_DIR = Path("../../gru/data")

# --- Load raw data ---
with open(DATA_DIR / "n8n-workflows.json") as f:
    wfs_raw = json.load(f)
with open(DATA_DIR / "n8n-mcp-mapping-summary.json") as f:
    mapping_data = json.load(f)
with open(DATA_DIR / "n8n-shgat-contrastive-pairs.json") as f:
    pairs = json.load(f)

mappings = mapping_data['mappings']
pair_by_id = {p['workflowId']: p for p in pairs}

# --- Topological sort + sequence extraction ---
EXCLUDED_SUFFIXES = {'Trigger', 'trigger', 'chatTrigger', 'manualTrigger', 'webhook',
    'if', 'switch', 'set', 'merge', 'code', 'noOp', 'stickyNote',
    'splitInBatches', 'wait', 'respondToWebhook', 'executeWorkflow',
    'function', 'functionItem', 'start'}

def is_plumbing(n8n_type):
    short = n8n_type.split('.')[-1] if '.' in n8n_type else n8n_type
    return short in EXCLUDED_SUFFIXES or 'Trigger' in short

def topo_sort(wf):
    edges = wf.get('edges', [])
    adj, in_deg, nodes = defaultdict(set), defaultdict(int), set()
    for e in edges:
        ft = e['fromType'] + (':' + e['fromOp'] if 'fromOp' in e else '')
        tt = e['toType'] + (':' + e['toOp'] if 'toOp' in e else '')
        if ft != tt:
            adj[ft].add(tt)
            in_deg[tt] = in_deg.get(tt, 0) + 1
            nodes.update([ft, tt])
            if ft not in in_deg: in_deg[ft] = 0
    queue = [n for n in nodes if in_deg.get(n, 0) == 0]
    result = []
    while queue:
        node = queue.pop(0)
        result.append(node)
        for nb in adj.get(node, []):
            in_deg[nb] -= 1
            if in_deg[nb] == 0: queue.append(nb)
    return result

# Build workflows with ordered tool sequences
workflows = []
for wf in wfs_raw:
    wid = wf['id']
    pair = pair_by_id.get(wid)
    if not pair or len(pair.get('intentEmbedding', [])) != 1024:
        continue
    sorted_n8n = topo_sort(wf)
    mcp_seq = []
    for key in sorted_n8n:
        if is_plumbing(key): continue
        m = mappings.get(key) or mappings.get(key.split(':')[0])
        if m and m['tier'] == 1 and m['sim'] >= 0.75:
            mcp_seq.append(m['mcp'])
    deduped = [mcp_seq[0]] if mcp_seq else []
    for t in mcp_seq[1:]:
        if t != deduped[-1]: deduped.append(t)
    if len(deduped) >= 2:
        workflows.append({
            'id': wid, 'name': wf.get('name', ''),
            'intent_emb': np.array(pair['intentEmbedding'], dtype=np.float32),
            'tool_seq': deduped, 'tool_set': set(deduped),
        })

# Tool vocabulary
all_tools = sorted(set(t for w in workflows for t in w['tool_seq']))
tool2idx = {t: i for i, t in enumerate(all_tools)}
N_TOOLS = len(all_tools)
STOP_IDX = N_TOOLS  # STOP token

wf_embs = np.array([w['intent_emb'] for w in workflows], dtype=np.float32)

print(f"Workflows with ordered sequences: {len(workflows)}")
print(f"Tools: {N_TOOLS}, Avg seq length: {np.mean([len(w['tool_seq']) for w in workflows]):.1f}")
print(f"Seq lengths: 2={sum(1 for w in workflows if len(w['tool_seq'])==2)}, "
      f"3={sum(1 for w in workflows if len(w['tool_seq'])==3)}, "
      f"4={sum(1 for w in workflows if len(w['tool_seq'])==4)}, "
      f"5+={sum(1 for w in workflows if len(w['tool_seq'])>=5)}")

In [ ]:
# === Tool-level directed transition graph (global) ===

tool_trans = defaultdict(lambda: defaultdict(int))
for w in workflows:
    seq = w['tool_seq']
    for i in range(len(seq) - 1):
        tool_trans[seq[i]][seq[i+1]] += 1

n_tool_edges = sum(len(v) for v in tool_trans.values())
print(f"Tool transition graph: {N_TOOLS} tools, {n_tool_edges} directed edges")
print(f"  Density: {n_tool_edges / N_TOOLS**2 * 100:.2f}%")
print(f"  Avg out-degree: {n_tool_edges / max(len(tool_trans), 1):.1f}")

# === Workflow-level directed graph builder ===

def build_directed_wf_graph(cat_wfs, tool_trans, min_evidence=1):
    """WF_i → WF_j if exit tools of WF_i transition to entry tools of WF_j."""
    # Inverted index: entry_tool → set of wf indices
    entries = defaultdict(set)
    for i, w in enumerate(cat_wfs):
        for t in w['tool_seq'][:2]:  # first 1-2 tools
            entries[t].add(i)

    adj = defaultdict(list)
    n_edges = 0
    for i, wi in enumerate(cat_wfs):
        exit_tools = wi['tool_seq'][-2:]  # last 1-2 tools
        candidates = defaultdict(int)
        for et in exit_tools:
            for succ, freq in tool_trans.get(et, {}).items():
                for j in entries.get(succ, set()):
                    if j != i:
                        candidates[j] += freq
        for j, ev in candidates.items():
            if ev >= min_evidence:
                adj[i].append((j, ev))
                n_edges += 1
    return adj, n_edges

# === Co-occurrence graph builder (for comparison) ===

def build_cooc_wf_graph(cat_wfs, threshold=0.05):
    """Undirected co-occurrence (Jaccard) — from nb03."""
    t2w = defaultdict(list)
    for i, w in enumerate(cat_wfs):
        for t in w['tool_set']:
            t2w[t].append(i)
    pairs = Counter()
    for tool, wf_idxs in t2w.items():
        for a in range(len(wf_idxs)):
            for b in range(a+1, len(wf_idxs)):
                pairs[(wf_idxs[a], wf_idxs[b])] += 1
    adj = defaultdict(list)
    n_edges = 0
    for (i, j), shared in pairs.items():
        union = len(cat_wfs[i]['tool_set'] | cat_wfs[j]['tool_set'])
        if union == 0: continue
        jacc = shared / union
        if jacc >= threshold:
            adj[i].append((j, jacc))
            adj[j].append((i, jacc))
            n_edges += 1
    return adj, n_edges

# === Compare at different scales ===
print("\n" + "="*65)
print("  GRAPH DENSITY COMPARISON: Directed vs Co-occurrence")
print("="*65)
print(f"  {'Size':>6s}  {'Dir edges':>10s}  {'Dir deg':>8s}  {'Dir isol':>9s}  {'Cooc edges':>11s}  {'Cooc deg':>9s}")
print(f"  {'-'*62}")

for size in [50, 200, 500, 1000]:
    if size > len(workflows): continue
    idx = np.random.RandomState(42).choice(len(workflows), size=size, replace=False)
    cat = [workflows[i] for i in idx]
    d_adj, d_e = build_directed_wf_graph(cat, tool_trans)
    c_adj, c_e = build_cooc_wf_graph(cat)
    d_degs = [len(d_adj.get(i, [])) for i in range(size)]
    c_degs = [len(c_adj.get(i, [])) for i in range(size)]
    d_isol = sum(1 for d in d_degs if d == 0)
    print(f"  {size:>6d}  {d_e:>10d}  {np.mean(d_degs):>7.1f}  "
          f"{d_isol:>4d}({d_isol/size*100:>3.0f}%)  {c_e:>11d}  {np.mean(c_degs):>8.1f}")

In [ ]:
# === All retrieval methods ===

def topk_cosine(intent, cat_embs, k=3):
    sims = cosine_similarity(intent.reshape(1, -1), cat_embs)[0]
    return list(np.argsort(-sims)[:k])

def iterative_residual(intent, cat_embs, max_steps=4, threshold=0.08):
    residual = intent.copy()
    selected, used = [], set()
    for _ in range(max_steps):
        sims = cosine_similarity(residual.reshape(1, -1), cat_embs)[0]
        for u in used: sims[u] = -1
        best = int(np.argmax(sims))
        if sims[best] < threshold: break
        selected.append(best); used.add(best)
        wn = cat_embs[best] / (np.linalg.norm(cat_embs[best]) + 1e-10)
        residual = residual - np.dot(residual, wn) * wn * 0.8
    return selected

def directed_graph_walk(intent, cat_embs, dir_adj, max_select=4, sim_floor=0.10):
    """Top-1 cosine → follow DIRECTED edges → greedy cosine among successors."""
    sims = cosine_similarity(intent.reshape(1, -1), cat_embs)[0]
    selected = [int(np.argmax(sims))]
    used = {selected[0]}
    for _ in range(max_select - 1):
        # Collect directed successors of all selected workflows
        candidates = {}
        for s in selected:
            for nbr, ev in dir_adj.get(s, []):
                if nbr not in used:
                    candidates[nbr] = max(candidates.get(nbr, 0), sims[nbr])
        if candidates:
            best = max(candidates, key=candidates.get)
            if candidates[best] >= sim_floor:
                selected.append(best); used.add(best); continue
        # Fallback: next best cosine (no graph neighbor available)
        for i in np.argsort(-sims):
            i = int(i)
            if i not in used:
                if sims[i] < sim_floor: break
                selected.append(i); used.add(i); break
        else:
            break
    return selected

def directed_graph_boosted(intent, cat_embs, dir_adj, max_select=4, boost=0.2, sim_floor=0.08):
    """Cosine + boost DIRECTED successors of selected workflows."""
    scores = cosine_similarity(intent.reshape(1, -1), cat_embs)[0].copy()
    selected, used = [], set()
    for _ in range(max_select):
        for u in used: scores[u] = -1
        best = int(np.argmax(scores))
        if scores[best] < sim_floor: break
        selected.append(best); used.add(best)
        for nbr, ev in dir_adj.get(best, []):
            if nbr not in used:
                scores[nbr] += boost * min(ev / 10, 1.0)  # normalize evidence
    return selected

def cooc_graph_walk(intent, cat_embs, cooc_adj, max_select=4, sim_floor=0.10):
    """Co-occurrence graph walk (from nb03 — for comparison)."""
    sims = cosine_similarity(intent.reshape(1, -1), cat_embs)[0]
    selected = [int(np.argmax(sims))]
    used = {selected[0]}
    for _ in range(max_select - 1):
        candidates = {}
        for s in selected:
            for nbr, w in cooc_adj.get(s, []):
                if nbr not in used:
                    candidates[nbr] = sims[nbr]
        if candidates:
            best = max(candidates, key=candidates.get)
            if candidates[best] >= sim_floor:
                selected.append(best); used.add(best); continue
        for i in np.argsort(-sims):
            i = int(i)
            if i not in used:
                if sims[i] < sim_floor: break
                selected.append(i); used.add(i); break
        else:
            break
    return selected

print("Methods: topk_cosine, iterative_residual, directed_graph_walk,")
print("         directed_graph_boosted, cooc_graph_walk")

In [ ]:
# === Tool-level Sequence GRU ===

class ToolSeqGRU(nn.Module):
    def __init__(self, intent_dim, n_tools, hidden_dim=128, tool_emb_dim=64):
        super().__init__()
        self.n_tools = n_tools
        self.STOP = n_tools
        self.hidden_dim = hidden_dim
        self.intent_proj = nn.Linear(intent_dim, hidden_dim)
        self.tool_emb = nn.Embedding(n_tools + 1, tool_emb_dim)  # +1 for STOP/SOS
        self.gru = nn.GRUCell(tool_emb_dim, hidden_dim)
        self.output = nn.Linear(hidden_dim, n_tools + 1)  # +1 for STOP

    def forward(self, intent, target_seq, max_steps=None):
        """Teacher-forced training. target_seq: (batch, seq_len) including STOP."""
        batch = intent.size(0)
        h = self.intent_proj(intent)
        steps = target_seq.size(1) if max_steps is None else max_steps
        sos = self.tool_emb(torch.full((batch,), self.STOP, dtype=torch.long, device=intent.device))
        prev = sos
        all_logits = []
        for t in range(steps):
            h = self.gru(prev, h)
            all_logits.append(self.output(h))
            if t < steps - 1:
                prev = self.tool_emb(target_seq[:, t])
        return all_logits

    def generate(self, intent, max_steps=12):
        """Greedy generation."""
        self.eval()
        with torch.no_grad():
            batch = intent.size(0)
            h = self.intent_proj(intent)
            prev = self.tool_emb(torch.full((batch,), self.STOP, dtype=torch.long, device=intent.device))
            preds = []
            for _ in range(max_steps):
                h = self.gru(prev, h)
                logits = self.output(h)
                pred = logits.argmax(dim=-1)
                preds.append(pred)
                if (pred == self.STOP).all(): break
                prev = self.tool_emb(pred)
            return torch.stack(preds, dim=1)

print(f"ToolSeqGRU defined. Vocab: {N_TOOLS} tools + STOP = {N_TOOLS+1}")

In [ ]:
# === Build training data: single + composite sequences ===

def build_gru_training_data(cat_wfs, cat_embs, dir_adj, tool2idx, STOP,
                            n_composite=2000, noise=0.08, seed=42):
    """
    Training data for tool-sequence GRU:
    1. Single workflows: (intent, [tool_ids..., STOP])
    2. Composite (graph-connected pairs): (mean_intent, [tools_A..., tools_B..., STOP])
    """
    rng = np.random.RandomState(seed)
    examples = []

    # --- Single workflow examples ---
    for i, w in enumerate(cat_wfs):
        idx_seq = [tool2idx[t] for t in w['tool_seq'] if t in tool2idx]
        if len(idx_seq) < 2: continue
        intent = cat_embs[i] + rng.randn(cat_embs.shape[1]).astype(np.float32) * noise
        examples.append((intent, idx_seq + [STOP], {i}))

    n_single = len(examples)

    # --- Composite: graph-connected pairs ---
    # Collect all directed edges
    all_edges = []
    for i, nbrs in dir_adj.items():
        for j, ev in nbrs:
            all_edges.append((i, j, ev))

    if all_edges:
        for _ in range(n_composite):
            src_i, src_j, _ = all_edges[rng.randint(len(all_edges))]
            seq_a = [tool2idx[t] for t in cat_wfs[src_i]['tool_seq'] if t in tool2idx]
            seq_b = [tool2idx[t] for t in cat_wfs[src_j]['tool_seq'] if t in tool2idx]
            if not seq_a or not seq_b: continue
            intent = (cat_embs[src_i] + cat_embs[src_j]) / 2
            intent += rng.randn(cat_embs.shape[1]).astype(np.float32) * noise
            examples.append((intent, seq_a + seq_b + [STOP], {src_i, src_j}))

    n_composite_actual = len(examples) - n_single
    rng.shuffle(examples)
    print(f"  Training data: {n_single} single + {n_composite_actual} composite = {len(examples)} total")
    return examples

print("build_gru_training_data defined")

In [ ]:
# === GRU Training ===

def train_gru(model, train_data, test_data, cat_embs, cat_wfs, tool2idx, STOP,
              epochs=25, batch_size=64, lr=0.001):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    max_seq = max(len(ex[1]) for ex in train_data)

    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        perm = np.random.permutation(len(train_data))
        epoch_loss, n_batches = 0, 0
        for bi in range(0, len(perm), batch_size):
            batch = [train_data[perm[j]] for j in range(bi, min(bi+batch_size, len(perm)))]
            intents = torch.tensor(np.array([b[0] for b in batch]), dtype=torch.float32)
            # Pad sequences
            padded = []
            for _, seq, _ in batch:
                p = seq[:max_seq]
                p = p + [STOP] * (max_seq - len(p))
                padded.append(p)
            targets = torch.tensor(padded, dtype=torch.long)
            logits_list = model(intents, targets)
            loss = sum(F.cross_entropy(lg, targets[:, t]) for t, lg in enumerate(logits_list)) / len(logits_list)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1

        # Quick eval
        model.eval()
        with torch.no_grad():
            test_intents = torch.tensor(np.array([t[0] for t in test_data]), dtype=torch.float32)
            preds = model.generate(test_intents, max_steps=12)
            idx2tool = {v: k for k, v in tool2idx.items()}
            n_correct = 0
            for j, (_, _, src_set) in enumerate(test_data):
                gen = [int(p) for p in preds[j].cpu().numpy() if p != STOP]
                gen_tools = set(idx2tool.get(g, '') for g in gen) - {''}
                # Match to catalog workflows by tool overlap
                matched = set()
                for wi, w in enumerate(cat_wfs):
                    if w['tool_set'] & gen_tools:
                        overlap = len(w['tool_set'] & gen_tools) / len(w['tool_set'])
                        if overlap >= 0.5:
                            matched.add(wi)
                if matched == src_set:
                    n_correct += 1
            seq_acc = n_correct / len(test_data) * 100

        elapsed = time.time() - t0
        avg_loss = epoch_loss / n_batches
        history.append({"epoch": epoch+1, "loss": avg_loss, "seq_acc": seq_acc})
        if (epoch+1) % 5 == 1 or epoch == epochs-1:
            print(f"  Epoch {epoch+1:>3d}  loss={avg_loss:.4f}  SeqAcc={seq_acc:.1f}%  [{elapsed:.1f}s]")

    return history

print("train_gru defined")

In [ ]:
# === Unified evaluation ===

def evaluate_all_methods(test_data, cat_embs_np, cat_wfs, dir_adj, cooc_adj, cat_size,
                         gru_model=None, tool2idx=None, STOP_IDX=None):
    methods = {
        "Top-1 cosine":  lambda i: topk_cosine(i, cat_embs_np, k=1),
        "Top-3 cosine":  lambda i: topk_cosine(i, cat_embs_np, k=3),
        "Residual":      lambda i: iterative_residual(i, cat_embs_np),
        "Cooc Walk":     lambda i: cooc_graph_walk(i, cat_embs_np, cooc_adj),
        "Dir Walk":      lambda i: directed_graph_walk(i, cat_embs_np, dir_adj),
        "Dir Boosted":   lambda i: directed_graph_boosted(i, cat_embs_np, dir_adj),
    }

    results = {}
    for name, method in methods.items():
        seq_accs, tool_rs, tool_ps = [], [], []
        for intent, _, src_set in test_data:
            pred = method(intent)
            true_t = set().union(*(cat_wfs[s]['tool_set'] for s in src_set))
            pred_t = set().union(*(cat_wfs[p]['tool_set'] for p in pred if 0 <= p < cat_size)) if pred else set()
            seq_accs.append(set(pred) == src_set)
            if true_t: tool_rs.append(len(true_t & pred_t) / len(true_t))
            if pred_t: tool_ps.append(len(true_t & pred_t) / len(pred_t))
        results[name] = {
            "seq_acc": np.mean(seq_accs) * 100,
            "tool_recall": np.mean(tool_rs) * 100,
            "tool_precision": np.mean(tool_ps) * 100 if tool_ps else 0,
        }

    # GRU evaluation
    if gru_model is not None:
        gru_model.eval()
        idx2tool = {v: k for k, v in tool2idx.items()}
        with torch.no_grad():
            t_intents = torch.tensor(np.array([t[0] for t in test_data]), dtype=torch.float32)
            preds = gru_model.generate(t_intents, max_steps=12)
        gru_sa, gru_tr, gru_tp = [], [], []
        for j, (_, _, src_set) in enumerate(test_data):
            gen = [int(p) for p in preds[j].cpu().numpy() if p != STOP_IDX]
            gen_tools = set(idx2tool.get(g, '') for g in gen) - {''}
            # Match generated tools to catalog workflows
            matched = set()
            for wi, w in enumerate(cat_wfs):
                overlap = len(w['tool_set'] & gen_tools)
                if overlap > 0 and overlap / len(w['tool_set']) >= 0.5:
                    matched.add(wi)
            true_t = set().union(*(cat_wfs[s]['tool_set'] for s in src_set))
            pred_t = gen_tools
            gru_sa.append(matched == src_set)
            if true_t: gru_tr.append(len(true_t & pred_t) / len(true_t))
            if pred_t: gru_tp.append(len(true_t & pred_t) / len(pred_t))
        results["GRU Seq"] = {
            "seq_acc": np.mean(gru_sa) * 100,
            "tool_recall": np.mean(gru_tr) * 100,
            "tool_precision": np.mean(gru_tp) * 100 if gru_tp else 0,
        }

    return results

def build_test_data(cat_embs, dir_adj, n_samples=500, max_compose=3, noise=0.10, seed=123):
    """Generate test intents: single + graph-connected composites."""
    rng = np.random.RandomState(seed)
    n_wf = len(cat_embs)
    examples = []
    all_edges = [(i, j) for i, nbrs in dir_adj.items() for j, _ in nbrs]

    for _ in range(n_samples):
        n_compose = rng.randint(1, max_compose + 1)
        if n_compose == 1 or not all_edges:
            src = [rng.randint(n_wf)]
        else:
            # Try to find graph-connected chain
            start_edge = all_edges[rng.randint(len(all_edges))]
            src = list(start_edge)
            if n_compose >= 3:
                # Try extending the chain
                last = src[-1]
                nbrs = [j for j, _ in dir_adj.get(last, []) if j not in src]
                if nbrs:
                    src.append(rng.choice(nbrs))
        intent = cat_embs[src].mean(axis=0) + rng.randn(cat_embs.shape[1]).astype(np.float32) * noise
        examples.append((intent, [], set(src)))
    return examples

print("evaluate_all_methods and build_test_data defined")

In [ ]:
# === SCALING EXPERIMENT ===

CATALOG_SIZES = [50, 200, 500, 1000]
all_results = {}

for cat_size in CATALOG_SIZES:
    if cat_size > len(workflows):
        print(f"\nSkip catalog={cat_size} (only {len(workflows)} available)")
        continue

    print(f"\n{'='*65}")
    print(f"  CATALOG = {cat_size}")
    print(f"{'='*65}")

    rng = np.random.RandomState(42)
    idx = rng.choice(len(workflows), size=cat_size, replace=False)
    cat = [workflows[i] for i in idx]
    cat_embs_np = wf_embs[idx]

    # Build graphs
    dir_adj, dir_e = build_directed_wf_graph(cat, tool_trans)
    cooc_adj, cooc_e = build_cooc_wf_graph(cat)
    dir_degs = [len(dir_adj.get(i, [])) for i in range(cat_size)]
    print(f"  Directed: {dir_e} edges, avg deg {np.mean(dir_degs):.1f}, "
          f"isolated {sum(1 for d in dir_degs if d==0)}")
    print(f"  Co-occur: {cooc_e} edges")

    # Build local tool2idx (only tools in this catalog)
    local_tools = sorted(set(t for w in cat for t in w['tool_seq']))
    local_t2i = {t: i for i, t in enumerate(local_tools)}
    local_stop = len(local_tools)

    # Training data for GRU
    train_ex = build_gru_training_data(cat, cat_embs_np, dir_adj, local_t2i, local_stop,
                                       n_composite=min(2000, dir_e * 3), noise=0.08)
    split = int(len(train_ex) * 0.85)
    train_set, val_set = train_ex[:split], train_ex[split:]

    # Train GRU
    gru = ToolSeqGRU(1024, len(local_tools), hidden_dim=128, tool_emb_dim=64).to(device)
    n_params = sum(p.numel() for p in gru.parameters())
    print(f"  GRU: {n_params:,} params, vocab={len(local_tools)}+STOP")
    hist = train_gru(gru, train_set, val_set, cat_embs_np, cat, local_t2i, local_stop,
                     epochs=20, batch_size=64)

    # Test data
    test = build_test_data(cat_embs_np, dir_adj, n_samples=500, seed=456)

    # Evaluate
    res = evaluate_all_methods(test, cat_embs_np, cat, dir_adj, cooc_adj, cat_size,
                               gru_model=gru, tool2idx=local_t2i, STOP_IDX=local_stop)
    all_results[cat_size] = res

    print(f"\n  {'Method':<16s}  {'SeqAcc':>7s}  {'ToolR':>7s}  {'ToolP':>7s}")
    print(f"  {'-'*44}")
    for name in ["Top-1 cosine", "Top-3 cosine", "Residual",
                 "Cooc Walk", "Dir Walk", "Dir Boosted", "GRU Seq"]:
        if name not in res: continue
        r = res[name]
        tag = " <DIR>" if "Dir" in name else (" <GRU>" if "GRU" in name else "")
        print(f"  {name:<16s}  {r['seq_acc']:>6.1f}%  {r['tool_recall']:>6.1f}%  "
              f"{r['tool_precision']:>6.1f}%{tag}")

In [ ]:
# === COMPOSITE INTENTS ONLY (2-3 workflows) ===

print("="*65)
print("COMPOSITE ONLY — Graph-connected multi-workflow intents")
print("="*65)

for cat_size in [200, 500, 1000]:
    if cat_size > len(workflows): continue
    rng = np.random.RandomState(42)
    idx = rng.choice(len(workflows), size=cat_size, replace=False)
    cat = [workflows[i] for i in idx]
    cat_embs_np = wf_embs[idx]

    dir_adj, _ = build_directed_wf_graph(cat, tool_trans)
    cooc_adj, _ = build_cooc_wf_graph(cat)

    # Generate composite-only test data
    test = build_test_data(cat_embs_np, dir_adj, n_samples=600, max_compose=3, seed=789)
    composite = [(i, t, s) for i, t, s in test if len(s) >= 2]
    if not composite:
        print(f"  Catalog={cat_size}: no composite intents (graph too sparse)")
        continue

    res = evaluate_all_methods(composite, cat_embs_np, cat, dir_adj, cooc_adj, cat_size)

    print(f"\n  Catalog={cat_size}, {len(composite)} composite intents:")
    print(f"  {'Method':<16s}  {'SeqAcc':>7s}  {'ToolR':>7s}  {'ToolP':>7s}")
    print(f"  {'-'*44}")
    for name in ["Top-1 cosine", "Top-3 cosine", "Residual",
                 "Cooc Walk", "Dir Walk", "Dir Boosted"]:
        r = res[name]
        is_best = r['seq_acc'] == max(res[n]['seq_acc'] for n in res)
        print(f"  {name:<16s}  {r['seq_acc']:>6.1f}%  {r['tool_recall']:>6.1f}%  "
              f"{r['tool_precision']:>6.1f}%{'  ***' if is_best else ''}")

In [ ]:
# === DELTA: Directed graph advantage ===

print("="*65)
print("DELTA ANALYSIS: Best Directed Graph - Best Flat (per scale)")
print("="*65)

flat_names = ["Top-1 cosine", "Top-3 cosine", "Residual"]
dir_names = ["Dir Walk", "Dir Boosted"]
cooc_names = ["Cooc Walk"]

print(f"  {'Size':>6s}  {'Best Flat':>14s} {'SA':>5s}  {'Best Dir':>12s} {'SA':>5s}  "
      f"{'Best Cooc':>10s} {'SA':>5s}  {'GRU':>5s}  {'Dir-Flat':>9s}")
print(f"  {'-'*80}")

for s in sorted(all_results.keys()):
    res = all_results[s]
    bf = max(flat_names, key=lambda m: res.get(m, {}).get('seq_acc', 0))
    bd = max(dir_names, key=lambda m: res.get(m, {}).get('seq_acc', 0))
    bc = max(cooc_names, key=lambda m: res.get(m, {}).get('seq_acc', 0))
    bf_sa = res.get(bf, {}).get('seq_acc', 0)
    bd_sa = res.get(bd, {}).get('seq_acc', 0)
    bc_sa = res.get(bc, {}).get('seq_acc', 0)
    gru_sa = res.get("GRU Seq", {}).get('seq_acc', 0)
    delta = bd_sa - bf_sa
    print(f"  {s:>6d}  {bf:>14s} {bf_sa:>4.1f}%  {bd:>12s} {bd_sa:>4.1f}%  "
          f"{bc:>10s} {bc_sa:>4.1f}%  {gru_sa:>4.1f}%  {'+' if delta>=0 else ''}{delta:>5.1f}pp")

print()
print("  Dir > Flat = directed graph structure helps")
print("  Dir >> Cooc = sequential edges >> co-occurrence")
print("  GRU = tool-sequence generation matched to workflows")

In [ ]:
# === Visualization ===
import matplotlib.pyplot as plt

sizes = sorted(all_results.keys())
method_order = ["Top-1 cosine", "Top-3 cosine", "Residual",
                "Cooc Walk", "Dir Walk", "Dir Boosted", "GRU Seq"]
colors = ['#4c72b0', '#55a868', '#c44e52', '#999999', '#8172b2', '#ccb974', '#e377c2']
markers = ['o', 's', '^', 'x', 'D', 'v', 'P']
styles = ['-', '-', '-', ':', '--', '--', '--']

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

for col, (metric, title) in enumerate([
    ('seq_acc', 'Seq Accuracy (%)'),
    ('tool_recall', 'Tool Recall (%)'),
    ('tool_precision', 'Tool Precision (%)')
]):
    ax = axes[col]
    for name, c, m, ls in zip(method_order, colors, markers, styles):
        vals = [all_results[s].get(name, {}).get(metric, 0) for s in sizes]
        if any(v > 0 for v in vals):
            ax.plot(sizes, vals, f'{m}{ls}', label=name, color=c, markersize=7, linewidth=2)
    ax.set_xlabel('Catalog Size')
    ax.set_ylabel('%')
    ax.set_title(title)
    ax.set_xscale('log')
    ax.axhline(y=80, color='red', linestyle=':', alpha=0.4)
    ax.legend(fontsize=7, loc='best')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 105)

plt.suptitle("Sequential Graph vs Flat vs Co-occurrence — Scaling", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('04-sequential-graph-scaling.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 04-sequential-graph-scaling.png")

## Conclusions

### Co-occurrence vs Directed Sequential vs GRU

| Aspect | Co-occurrence graph | Directed sequential graph | Tool-level GRU |
|---|---|---|---|
| **Edges** | ~millions (dense) | ~milliers (sparse) | N/A (learned) |
| **Direction** | Non | Oui (A→B) | Implicite (sequence) |
| **Signal** | "Apparaissent ensemble" | "A suit B dans le pipeline" | Apprend l'ordre |
| **Scaling** | Quasi-clique partout | Structure conservee | Retrain necessaire |

### Quand utiliser quoi (previsions)

| Taille catalog | Recommandation |
|----------------|----------------|
| < 100 | Cosine top-1 suffit |
| 100-500 | Directed graph walk |
| 500+ | GRU + directed graph |

### Implications pour l'app industrielle

1. **Construire le graph de transitions** entre outils au fur et a mesure (pas de co-occurrence)
2. **Le graph est un produit** des workflows sauvegardes — chaque nouveau workflow enrichit le graph
3. **GRU optionnel** — le graph walk dirige suffit si les workflows sont bien sequences
4. **Cout zero** — les methodes graph n'ont pas besoin d'entrainement